In [119]:
# CELL 1 — Install dependencies
import sys
!{sys.executable} -m pip install -q easyocr rapidfuzz pandas numpy opencv-python pillow matplotlib pymupdf imagehash paddleocr

In [120]:
# CELL 2 — Imports
from pathlib import Path
import json, re, zipfile, math
from collections import Counter
import numpy as np
import pandas as pd
import cv2
import fitz
import easyocr
from rapidfuzz import fuzz, process
from PIL import Image
import imagehash
import matplotlib.pyplot as plt

try:
    from paddleocr import PaddleOCR
    PADDLE_AVAILABLE = True
except Exception:
    PADDLE_AVAILABLE = False

print('Imports OK | Paddle available =', PADDLE_AVAILABLE)

Imports OK | Paddle available = True


In [121]:
# CELL 3 — Config
INPUT_PATH = ".\\phase_a_v3_full_suite\\raw\\PHARMACY_BILL\\pharmacy_bill_printed_003.png"  # Path to input document (image or PDF)
EXPECTED_TAG = "PHARMACY_BILL"  # PRESCRIPTION, HOSPITAL_BILL, LAB_REPORT, PHARMACY_BILL
OCR_MODE = "AUTO" # AUTO, EASYOCR, PADDLE

RXNORM_ZIP = "./reference_data/RxNorm_full_prescribe_current.zip"
RXNORM_DIR = "./reference_data/rxnorm_extracted"
DIAGNOSIS_CSV = "./reference_data/diagnosis_vocab.csv"
LABTEST_CSV = "./reference_data/labtest_vocab.csv"
DOC_TYPE = 'AUTO'


OUT_DIR = Path('./complete_pipeline_v3_output')
PAGES_DIR = OUT_DIR / "pages"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PAGES_DIR.mkdir(parents=True, exist_ok=True)

print('INPUT_PATH =', INPUT_PATH)
print('EXPECTED_TAG =', EXPECTED_TAG)


INPUT_PATH = .\phase_a_v3_full_suite\raw\PHARMACY_BILL\pharmacy_bill_printed_003.png
EXPECTED_TAG = PHARMACY_BILL


In [122]:
# CELL 4 — Render input pages
def render_pdf_pages(pdf_path, dpi=220):
    pages = []
    doc = fitz.open(pdf_path)
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=dpi, alpha=False)
        out = PAGES_DIR / f'page_{i+1:03d}.png'
        pix.save(str(out))
        pages.append(str(out))
    return pages

input_path = Path(INPUT_PATH)
page_files = render_pdf_pages(str(input_path)) if input_path.suffix.lower() == '.pdf' else [str(input_path)]
print('Pages:', len(page_files))

Pages: 1


In [123]:
# CELL 5 — OCR engines
easy_reader = easyocr.Reader(['en'], gpu=False, verbose=False)
paddle_reader = None
if PADDLE_AVAILABLE:
    try:
        paddle_reader = PaddleOCR(use_angle_cls=True, lang='en')
    except Exception:
        paddle_reader = None

print('EasyOCR ready')
print('Paddle ready =', paddle_reader is not None)

C:\Users\gokul\AppData\Local\Temp\ipykernel_3068\343390615.py:6: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  paddle_reader = PaddleOCR(use_angle_cls=True, lang='en')
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\gokul\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\gokul\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\gokul\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cach

EasyOCR ready
Paddle ready = True


In [124]:
# CELL 6 — Dictionaries and validation rules
SKIP_PREFIXES = ['age','gender','date','diagnosis','patient','chief complaint','follow-up','follow up','signed','name','ph','phone','hospital','investigations','advice','address','reg no','registration','doctor','page','bill to']
MED_HINTS = ['tab','tablet','cap','capsule','syp','syrup','inj','injection','cream','ointment','gel','drops','drop','lotion','spray']
STAMP_TERMS = ['original','duplicate','paid','cash','cancelled','received']
SPECIALIZATION_TERMS = ['mbbs','md','ms','dm','mch','dnb','internal medicine','dermatology','pathology','pediatrics','paediatrics','orthopedics','orthopaedics','gynaecology','ent','cardiology','neurology','psychiatry','general medicine']
DIAGNOSIS_SHORTHAND = {
    'htn': 'Hypertension', 'hbp': 'Hypertension', 't2dm': 'Type 2 Diabetes', 't1dm': 'Type 1 Diabetes',
    'dm': 'Diabetes Mellitus', 'gerd': 'GERD', 'uti': 'UTI', 'uri': 'URTI', 'copd': 'COPD Exacerbation',
    'oa': 'Osteoarthritis', 'ra': 'Rheumatoid Arthritis', 'cap': 'Community Acquired Pneumonia',
    'lbp': 'Back Pain', 'migraine': 'Migraine', 'ibs': 'IBS', 'pud': 'Peptic Ulcer Disease', 'hypothy': 'Hypothyroidism'
}
REG_PATTERNS = {
    'KA': r'KA/\d{4,6}/\d{4}', 'MH': r'MH/\d{4,6}/\d{4}', 'DL': r'DL/\d{4,6}/\d{4}',
    'TN': r'TN/\d{4,6}/\d{4}', 'GJ': r'GJ/\d{4,6}/\d{4}', 'AP': r'AP/\d{4,6}/\d{4}',
    'UP': r'UP/\d{4,6}/\d{4}', 'WB': r'WB/\d{4,6}/\d{4}', 'KL': r'KL/\d{4,6}/\d{4}',
    'AYUR': r'AYUR/[A-Z]{2}/\d{3,6}/\d{4}'
}

def nk(x):
    return re.sub(r'[^a-z0-9]+', ' ', str(x or '').lower()).strip()

def normalize_ocr_noise(text):
    text = str(text or '')
    text = text.replace('–','-').replace('—','-')
    text = re.sub(r'(?<=\d)[oO](?=\d|\s*(mg|mcg|ml|g|gm|iu|rs))', '0', text)
    text = re.sub(r'(?<=\d)[lI](?=\d)', '1', text)
    text = re.sub(r'\b1X\b', 'x', text, flags=re.I)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def expand_diagnosis_shorthand(text):
    tokens = nk(text).split()
    expanded = [DIAGNOSIS_SHORTHAND.get(t, t) for t in tokens]
    return ' '.join(expanded).strip()

def validate_reg_no(text):
    t = normalize_ocr_noise(text)
    for state, pat in REG_PATTERNS.items():
        m = re.search(pat, t)
        if m:
            return {'value': m.group(0), 'state': state, 'valid_format': True}
    generic = re.search(r'[A-Z]{2}/\d{4,6}/\d{4}', t)
    if generic:
        return {'value': generic.group(0), 'state': generic.group(0).split('/')[0], 'valid_format': False}
    return None

In [125]:
# CELL 7 — OCR preprocessing and engine routing
def detect_script_flag(text):
    if re.search(r'[\u0900-\u097F]', text): return 'DEVANAGARI'
    if re.search(r'[\u0B80-\u0BFF]', text): return 'TAMIL'
    if re.search(r'[\u0C00-\u0C7F]', text): return 'TELUGU'
    if re.search(r'[\u0C80-\u0CFF]', text): return 'KANNADA'
    return None


def estimate_quality_flags(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    brightness = float(np.mean(gray)); contrast = float(np.std(gray))
    edges = cv2.Canny(gray, 100, 200)
    edge_density = float(np.mean(edges > 0))
    flags = []
    if brightness < 70: flags.append('LOW_BRIGHTNESS')
    if contrast < 35: flags.append('LOW_CONTRAST')
    if edge_density < 0.01: flags.append('BLUR_OR_LOW_TEXT_DENSITY')
    return {'brightness': round(brightness, 2), 'contrast': round(contrast, 2), 'edge_density': round(edge_density, 4), 'flags': flags}


def deskew_image(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    inv = cv2.bitwise_not(gray)
    coords = np.column_stack(np.where(inv > 0))
    if len(coords) < 50:
        return img_bgr, 0.0
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) > 20:
        return img_bgr, 0.0
    h, w = img_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    rotated = cv2.warpAffine(img_bgr, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return rotated, float(angle)


def preprocess_for_ocr(img_bgr):
    rot, angle = deskew_image(img_bgr)
    gray = cv2.cvtColor(rot, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    denoise = cv2.fastNlMeansDenoising(clahe, None, 8, 7, 21)
    denoise_bgr = cv2.cvtColor(denoise, cv2.COLOR_GRAY2BGR)
    denoise_rgb = cv2.cvtColor(denoise_bgr, cv2.COLOR_BGR2RGB)
    return denoise, denoise_bgr, denoise_rgb, angle


def choose_ocr_mode(quality):
    if OCR_MODE in ['EASYOCR', 'PADDLE']:
        return OCR_MODE
    hard = ('LOW_CONTRAST' in quality['flags']) or ('BLUR_OR_LOW_TEXT_DENSITY' in quality['flags'])
    if hard and paddle_reader is not None:
        return 'EASYOCR'
    return 'EASYOCR'


def normalize_paddle_output(res):
    raw_results = []

    if res is None:
        return raw_results

    for page in res:
        if page is None:
            continue

        if isinstance(page, dict):
            data = page.get('res', page)

            polys = data.get('dt_polys') or data.get('rec_polys') or data.get('rec_boxes')
            texts = data.get('rec_texts')
            scores = data.get('rec_scores')

            if polys is not None and texts is not None and scores is not None:
                for box, text, score in zip(polys, texts, scores):
                    raw_results.append((
                        box.tolist() if hasattr(box, 'tolist') else box,
                        str(text),
                        float(score)
                    ))
                continue

        if isinstance(page, list):
            for item in page:
                if item is None:
                    continue

                if isinstance(item, (list, tuple)):
                    if len(item) == 2:
                        box, txt = item
                        if isinstance(txt, (list, tuple)) and len(txt) >= 2:
                            raw_results.append((box, str(txt[0]), float(txt[1])))
                        else:
                            raw_results.append((box, str(txt), 0.0))

                    elif len(item) >= 3:
                        box = item[0]
                        text = item[1]
                        score = item[2]

                        if isinstance(text, (list, tuple)) and len(text) >= 1:
                            text = text[0]
                        if isinstance(score, (list, tuple)) and len(score) >= 1:
                            score = score[0]

                        try:
                            score = float(score)
                        except:
                            score = 0.0

                        raw_results.append((box, str(text), score))

    return raw_results


def run_ocr(page_path):
    img_bgr = cv2.imread(page_path)
    assert img_bgr is not None, f'Could not read {page_path}'

    quality = estimate_quality_flags(img_bgr)
    ocr_gray, ocr_bgr, ocr_rgb, angle = preprocess_for_ocr(img_bgr)
    mode = choose_ocr_mode(quality)
    print('mode: ', mode)

    if mode == 'PADDLE' and paddle_reader is not None:
        try:
            res = paddle_reader.ocr(ocr_rgb)
            raw_results = normalize_paddle_output(res)

            if not raw_results:
                mode = 'EASYOCR'
                raw_results = easy_reader.readtext(ocr_gray)
                ocr_input = ocr_gray
            else:
                ocr_input = ocr_rgb

        except Exception:
            print('Got exception with PADDLE')
            mode = 'EASYOCR'
            raw_results = easy_reader.readtext(ocr_gray)
            ocr_input = ocr_gray

    else:
        print('Using EasyOCR directly in else block')
        mode = 'EASYOCR'
        raw_results = easy_reader.readtext(ocr_gray)
        ocr_input = ocr_gray

    return img_bgr, ocr_input, angle, quality, mode, raw_results

In [126]:
# CELL 8 — OCR to lines
def ocr_to_lines(results, y_tol=16):
    words = []
    for box, text, conf in results:
        xs = [p[0] for p in box]
        ys = [p[1] for p in box]
        words.append({'text': normalize_ocr_noise(text), 'confidence': float(conf), 'x1': float(min(xs)), 'y1': float(min(ys)), 'x2': float(max(xs)), 'y2': float(max(ys))})
    words = sorted(words, key=lambda r: (r['y1'], r['x1']))
    grouped, current, anchor = [], [], None
    for w in words:
        if anchor is None or abs(w['y1'] - anchor) <= y_tol:
            current.append(w)
        else:
            grouped.append(sorted(current, key=lambda r: r['x1']))
            current = [w]
        anchor = w['y1'] if anchor is None else (anchor + w['y1']) / 2
    if current:
        grouped.append(sorted(current, key=lambda r: r['x1']))
    lines = []
    for row in grouped:
        txt = ' '.join(r['text'] for r in row).strip()
        lines.append({'text': txt, 'confidence': float(np.mean([r['confidence'] for r in row])), 'x1': min(r['x1'] for r in row), 'y1': min(r['y1'] for r in row), 'x2': max(r['x2'] for r in row), 'y2': max(r['y2'] for r in row), 'parts': row})
    return lines

In [127]:
# CELL 9 — Standards-based vocab loads
SAFE_TTYS = {'IN','PIN','MIN','SCD','SBD','SCDC','SCDF','SBDC'}

def read_rxnconso_from_zip(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        name = [n for n in zf.namelist() if n.upper().endswith('RXNCONSO.RRF')][0]
        with zf.open(name) as f:
            df = pd.read_csv(f, sep='|', header=None, dtype=str, encoding='utf-8', engine='python')
    return df

def read_rxnconso_from_dir(folder):
    file_path = list(Path(folder).rglob('RXNCONSO.RRF'))[0]
    return pd.read_csv(file_path, sep='|', header=None, dtype=str, encoding='utf-8', engine='python')

rx_df = read_rxnconso_from_zip(RXNORM_ZIP) if Path(RXNORM_ZIP).exists() else read_rxnconso_from_dir(RXNORM_DIR)
rx_df = rx_df.iloc[:, :18].copy()
rx_df.columns = ['RXCUI','LAT','TS','LUI','STT','SUI','ISPREF','RXAUI','SAUI','SCUI','SDUI','SAB','TTY','CODE','STR','SRL','SUPPRESS','CVF']
rx_df = rx_df[(rx_df['LAT']=='ENG') & (rx_df['STR'].notna()) & (rx_df['SAB']=='RXNORM') & (rx_df['TTY'].isin(SAFE_TTYS))]
rx_df['STR_NORM'] = rx_df['STR'].str.lower().str.replace(r'[^a-z0-9]+', ' ', regex=True).str.strip()
rx_df = rx_df[rx_df['STR_NORM'].str.len() >= 4].drop_duplicates(subset=['RXCUI','STR_NORM']).reset_index(drop=True)

def load_vocab_csv(path):
    p = Path(path)
    if not p.exists():
        return pd.DataFrame(columns=['canonical','code','alias','alias_norm'])
    df = pd.read_csv(p, dtype=str).fillna('')
    df['alias_norm'] = df['alias'].str.lower().str.replace(r'[^a-z0-9]+', ' ', regex=True).str.strip()
    return df

diag_df = load_vocab_csv(DIAGNOSIS_CSV)
lab_df = load_vocab_csv(LABTEST_CSV)
print('RxNorm rows:', len(rx_df), '| diagnosis rows:', len(diag_df), '| lab rows:', len(lab_df))

RxNorm rows: 51490 | diagnosis rows: 0 | lab rows: 0


In [128]:
# CELL 10 — Sections, doc type, and matching
SECTION_PATTERNS = {
    'diagnosis': [r'^diagnosis[:\s]*$', r'^dx[:\s]*$', r'^impression[:\s]*$'],
    'rx': [r'^rx[:\s]*$', r'^treatment[:\s]*$', r'^medicines?[:\s]*$'],
    'tests': [r'^investigations?[:\s]*$', r'^tests? advised[:\s]*$', r'^labs?[:\s]*$'],
    'followup': [r'^follow[- ]?up[:\s]*$', r'^advice[:\s]*$']
}

def infer_doc_type(lines):
    txt = ' '.join(normalize_ocr_noise(x['text']).lower() for x in lines)
    scores = Counter()
    if 'rx' in txt or 'chief complaint' in txt or 'diagnosis' in txt: scores['PRESCRIPTION'] += 3
    if 'bill no' in txt or 'invoice' in txt or 'payment mode' in txt or 'subtotal' in txt: scores['HOSPITAL_BILL'] += 3
    if 'drug lic' in txt or 'net amount' in txt or 'pharmacy' in txt or 'mrp' in txt or 'batch' in txt: scores['PHARMACY_BILL'] += 3
    if 'sample id' in txt or 'reference range' in txt or 'lab report' in txt or 'haemoglobin' in txt or 'normal range' in txt: scores['LAB_REPORT'] += 3
    return scores.most_common(1)[0][0] if scores else 'PRESCRIPTION'

def detect_section(line_text):
    t = nk(line_text)
    for section, patterns in SECTION_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, t, re.I): return section
    return None

def split_inline_section(line_text):
    raw = normalize_ocr_noise(line_text)
    for label in ['Diagnosis','Dx','Impression','Rx','Treatment','Medicines','Investigations','Investigation','Tests Advised','Labs','Follow-up','Follow up','Advice']:
        m = re.match(rf'^{label}[:\s]+(.+)$', raw, flags=re.I)
        if m: return label.lower(), m.group(1).strip()
    return None, None

def build_sections(lines):
    sections = {'header': [], 'diagnosis': [], 'rx': [], 'tests': [], 'followup': [], 'other': []}
    current = 'header'
    for ln in lines:
        txt = normalize_ocr_noise(ln['text'])
        sec = detect_section(txt)
        if sec:
            current = sec
            continue
        inline_key, inline_val = split_inline_section(txt)
        if inline_key:
            target = 'diagnosis' if inline_key in ['diagnosis','dx','impression'] else 'rx' if inline_key in ['rx','treatment','medicines'] else 'tests' if inline_key.startswith('investigation') or inline_key.startswith('tests') or inline_key == 'labs' else 'followup'
            new_ln = dict(ln); new_ln['text'] = inline_val
            sections[target].append(new_ln)
            current = target
            continue
        sections[current if current in sections else 'other'].append(ln)
    return sections

def looks_like_medicine_line(text):
    t = normalize_ocr_noise(text); k = nk(t)
    if len(k) < 5 or any(k.startswith(p) for p in SKIP_PREFIXES): return False
    if any(h in k.split() for h in MED_HINTS): return True
    if re.search(r'\b\d+(?:\.\d+)?\s*(mg|mcg|ml|g|gm|iu|units?)\b', t, re.I): return True
    if re.search(r'\b(OD|BD|TDS|QID|SOS|PRN|HS|AC|PC)\b|\b\d-\d-\d\b', t, re.I): return True
    if re.search(r'^\d+[.)]', t): return True
    return False

def best_rxnorm_match(text, threshold_main=90, threshold_loose=86):
    raw = normalize_ocr_noise(text); key = nk(raw)
    if not key or len(key) < 5 or not looks_like_medicine_line(raw): return None
    core = re.sub(r'^\d+[.)]\s*', '', raw)
    core = re.sub(r'\b(tab|tablet|cap|capsule|syp|syrup|inj|injection|cream|ointment|gel|drops?|lotion|spray)\b', ' ', core, flags=re.I)
    core = re.sub(r'\b(od|bd|tds|qid|sos|prn|hs|ac|pc)\b', ' ', core, flags=re.I)
    core = re.sub(r'\b\d-\d-\d\b', ' ', core)
    core = re.sub(r'\bx\s*\d+\s*(days?|weeks?|months?)\b', ' ', core, flags=re.I)
    core_key = nk(core) or key
    candidates = rx_df['STR_NORM'].tolist()
    scorer = fuzz.WRatio if len(core_key.split()) <= 3 else fuzz.token_set_ratio
    result = process.extractOne(core_key, candidates, scorer=scorer)
    if not result: return None
    threshold = threshold_loose if re.search(r'\b\d+(?:\.\d+)?\s*(mg|mcg|ml|g|gm|iu)\b', raw, re.I) else threshold_main
    if result[1] < threshold: return None
    row = rx_df.iloc[result[2]]
    overlap = set(core_key.split()).intersection(set(str(row['STR_NORM']).split()))
    if len(overlap) == 0: return None
    return {'canonical': row['STR'], 'rxcui': row['RXCUI'], 'tty': row['TTY'], 'sab': row['SAB'], 'score': float(result[1]), 'method': 'rxnorm_safe_fuzzy', 'core_query': core_key}

def best_vocab_match(text, vocab_df, threshold=78, label='vocab'):
    raw = expand_diagnosis_shorthand(normalize_ocr_noise(text)) if label == 'diagnosis' else normalize_ocr_noise(text)
    key = nk(raw)
    if not key or vocab_df.empty: return None
    candidates = vocab_df['alias_norm'].tolist()
    scorer = fuzz.WRatio if len(key.split()) <= 3 else fuzz.token_set_ratio
    result = process.extractOne(key, candidates, scorer=scorer)
    if not result or result[1] < threshold: return None
    row = vocab_df.iloc[result[2]]
    return {'canonical': row['canonical'], 'code': row['code'], 'alias': row['alias'], 'score': float(result[1]), 'method': f'{label}_safe_fuzzy'}

In [129]:
# CELL 11 — Richer field extraction and table parsing
def find_first(lines, patterns):
    for ln in lines:
        for pat in patterns:
            m = re.search(pat, normalize_ocr_noise(ln['text']), re.I)
            if m: return m, ln
    return None, None

def named_entity(lines, prefixes):
    for ln in lines:
        tx = normalize_ocr_noise(ln['text'])
        for p in prefixes:
            if nk(tx).startswith(nk(p)):
                val = tx.split(':',1)[1].strip() if ':' in tx else tx[len(p):].strip()
                return {'value': val, 'confidence': round(float(ln['confidence']),3)}
    return None

def extract_specialization(header_lines):
    matches = []
    for ln in header_lines[:8]:
        t = nk(ln['text'])
        found = [s for s in SPECIALIZATION_TERMS if s in t]
        if found:
            matches.extend(found)
    return sorted(list(set(matches)))

def extract_address(lines):
    address_lines = []
    for ln in lines[:6]:
        t = normalize_ocr_noise(ln['text'])
        if re.search(r'road|rd\b|street|st\b|nagar|layout|bengaluru|bangalore|chennai|mumbai|delhi|560\d{3}|\d{6}', t, re.I):
            address_lines.append(t)
    return ' | '.join(address_lines) if address_lines else None

def extract_gstin(lines):
    m, ln = find_first(lines, [r'\b\d{2}[A-Z]{5}\d{4}[A-Z]\d[Z][A-Z0-9]\b'])
    return {'value': m.group(0), 'confidence': round(float(ln['confidence']),3)} if m else None

def extract_nabl(lines):
    for ln in lines[:8]:
        if re.search(r'nabl', ln['text'], re.I):
            return {'value': normalize_ocr_noise(ln['text']), 'confidence': round(float(ln['confidence']),3)}
    return None

def parse_hospital_bill_rows(lines):
    """
    Robust parser for hospital bill line items.

    Handles layouts where DESCRIPTION / QTY / RATE / AMOUNT appear in a header row,
    and then the numeric values and descriptions are split across multiple lines
    (typical EasyOCR output).

    Returns a list of dicts: {description, qty, rate, amount}.
    """

    # 1) Identify table region boundaries: header row, then stop at footer
    header_idx = None
    footer_idx = len(lines)

    header_pattern = re.compile(r"description", re.I)
    footer_patterns = re.compile(
        r"subtotal|total amount|gst|payment mode|amount paid|balance",
        re.I,
    )

    for i, ln in enumerate(lines):
        t = normalize_ocr_noise(ln["text"])
        if header_idx is None and header_pattern.search(t):
            header_idx = i
            continue
        if header_idx is not None and footer_patterns.search(t):
            footer_idx = i
            break

    # If no recognizable header, bail out (let doc-level validation catch missing line_items)
    if header_idx is None:
        return []

    body = lines[header_idx + 1:footer_idx]

    # 2) Classify lines in body as numeric-only vs description
    numeric_lines = []   # (index_in_body, numeric_value)
    desc_lines = []      # (index_in_body, cleaned_text)

    # Strict numeric pattern: full line is something like 500 or 500.00
    num_pat = re.compile(r"^[0-9]+(?:\.[0-9]{1,2})?$")

    for idx, ln in enumerate(body):
        t = normalize_ocr_noise(ln["text"]).strip()
        if not t:
            continue
        if num_pat.match(t):
            try:
                numeric_lines.append((idx, float(t)))
            except Exception:
                # If float conversion fails, treat as description
                desc_lines.append((idx, t))
        else:
            desc_lines.append((idx, t))

    if not numeric_lines or not desc_lines:
        return []

    # 3) Pair numeric pairs with nearby description lines
    items = []
    used_numeric_idx = set()

    def find_nearest_desc(i1, i2):
        """
        Find a description line whose index is between i1 and i2,
        or, if none, the closest one in terms of index distance.
        """
        if not desc_lines:
            return None
        candidates = []
        for di, txt in desc_lines:
            # Prefer descriptions between the two numeric indices
            if i1 <= di <= i2 or i2 <= di <= i1:
                # Distance 0 means inside [i1, i2]
                candidates.append((0, di, txt))
            else:
                # Distance to closest numeric index
                d = min(abs(di - i1), abs(di - i2))
                candidates.append((d, di, txt))
        # Sort by (distance, index) to ensure deterministic choice
        candidates.sort(key=lambda x: (x[0], x[1]))
        return candidates[0][1], candidates[0][2]

    i = 0
    while i < len(numeric_lines) - 1:
        idx1, v1 = numeric_lines[i]
        idx2, v2 = numeric_lines[i + 1]

        # Skip if already used in another pair
        if idx1 in used_numeric_idx or idx2 in used_numeric_idx:
            i += 1
            continue

        desc_info = find_nearest_desc(idx1, idx2)
        if not desc_info:
            i += 1
            continue

        desc_idx, desc_txt = desc_info

        # Filter out obvious footer-like descriptions accidentally captured
        if re.search(r"subtotal|total amount|gst|payment mode", desc_txt, re.I):
            i += 1
            continue

        # Build the item (assume qty=1 if not explicitly present)
        items.append(
            {
                "description": desc_txt,
                "qty": 1,
                "rate": v1,
                "amount": v2,
            }
        )

        used_numeric_idx.add(idx1)
        used_numeric_idx.add(idx2)
        i += 2

    return items


def parse_pharmacy_rows(lines):
    """
    Robust parser for pharmacy bill line items.

    Handles layouts where there is a header like:
        BATCH
        MEDICINE
        EXP QTY MRP AMT

    and each item is represented as:
        <medicine name>
        <BATCH> <EXP> <QTY> <MRP> <AMOUNT>

    Returns list of dicts:
        {
            "medicine": str,
            "batch": str or None,
            "expiry": str or None,
            "qty": int or None,
            "mrp": float or None,
            "amount": float or None,
        }
    """

    # 1) Identify table region (header -> footer)
    header_idx = None
    footer_idx = len(lines)

    # Look for the composite header: EXP QTY MRP AMT
    header_pattern = re.compile(r"\bEXP\b.*\bMRP\b.*\bAMT\b", re.I)
    footer_patterns = re.compile(
        r"subtotal|net amount|payment mode|gst|total amount|amount paid|balance",
        re.I,
    )

    for i, ln in enumerate(lines):
        t = normalize_ocr_noise(ln["text"])
        if header_idx is None and header_pattern.search(t):
            header_idx = i
            continue
        if header_idx is not None and footer_patterns.search(t):
            footer_idx = i
            break

    # If we didn't find the EXP/QTY/MRP/AMT header, bail out
    if header_idx is None:
        return []

    body = lines[header_idx + 1:footer_idx]

    # 2) Walk the body as (name_line, detail_line) pairs
    items = []

    # Pattern for detail line: BATCH EXP QTY MRP AMT
    # Example: "B7821 06/26 13 5.50 71.50"
    detail_pat = re.compile(
        r"""
        ^\s*
        (?P<batch>[A-Z0-9]{3,10})        # batch code, e.g. B7821
        \s+
        (?P<exp>\d{2}/\d{2,4})          # expiry MM/YY or MM/YYYY
        \s+
        (?P<qty>\d{1,4})                # quantity
        \s+
        (?P<mrp>\d+(?:\.\d{1,2})?)      # MRP
        \s+
        (?P<amt>\d+(?:\.\d{1,2})?)      # amount
        \s*$
        """,
        re.VERBOSE,
    )

    i = 0
    while i < len(body):
        # Expect medicine name on line i, detail on line i+1
        name_line = body[i]
        name_text = normalize_ocr_noise(name_line["text"]).strip()

        if not name_text:
            i += 1
            continue

        # No detail line => we can't parse this item
        if i + 1 >= len(body):
            break

        detail_line = body[i + 1]
        detail_text = normalize_ocr_noise(detail_line["text"]).strip()

        m = detail_pat.match(detail_text)
        if not m:
            # If the detail line doesn't match, skip this pair
            i += 1
            continue

        batch = m.group("batch")
        exp = m.group("exp")

        qty_str = m.group("qty")
        mrp_str = m.group("mrp")
        amt_str = m.group("amt")

        qty = int(qty_str) if qty_str is not None else None
        try:
            mrp = float(mrp_str) if mrp_str is not None else None
        except Exception:
            mrp = None
        try:
            amt = float(amt_str) if amt_str is not None else None
        except Exception:
            amt = None

        items.append(
            {
                "medicine": name_text,
                "batch": batch,
                "expiry": exp,
                "qty": qty,
                "mrp": mrp,
                "amount": amt,
            }
        )

        # consume name + detail
        i += 2

    return items



def parse_lab_rows(lines):
    rows = []
    for ln in lines:
        t = normalize_ocr_noise(ln['text'])
        if re.search(r'test name|normal range|sample id|report date|remarks|pathology', t, re.I):
            continue
        m = re.match(r'(.+?)\s+([A-Za-z0-9,./+-]+)\s+([/%A-Za-zμdL]+)\s+([0-9.,\-– ]+)$', t)
        if m:
            rows.append({'test_name': m.group(1).strip(), 'result': m.group(2), 'unit': m.group(3), 'normal_range': m.group(4).strip()})
        elif re.search(r'negative|positive|reactive|non reactive', t, re.I):
            parts = re.split(r'\s{2,}|	', t)
            if len(parts) >= 2:
                rows.append({'test_name': parts[0].strip(), 'result': parts[1].strip(), 'unit': '', 'normal_range': ''})
    return rows

In [130]:
# CELL 12 — Fraud and page checks
def page_fingerprint(page_path):
    return str(imagehash.phash(Image.open(page_path).convert('RGB')))

def detect_duplicate_stamps(lines):
    hits = []
    for ln in lines:
        t = nk(ln['text'])
        for s in STAMP_TERMS:
            if s in t: hits.append(s.upper())
    counts = Counter(hits)
    return [k for k, v in counts.items() if v >= 2]

def detect_document_alteration(lines):
    text = ' '.join(normalize_ocr_noise(x['text']).lower() for x in lines)
    cues = []
    if re.search(r'corrected|revised|overwritten|rewritten', text): cues.append('TEXTUAL_CORRECTION_CUE')
    if len(re.findall(r'\b(total|amount|net amount|subtotal)\b', text)) >= 2: cues.append('MULTIPLE_AMOUNT_MENTIONS')
    return cues

def detect_partial_document(lines, image_shape):
    h, w = image_shape[:2]
    if not lines: return ['EMPTY_PAGE']
    xs = [x['x1'] for x in lines] + [x['x2'] for x in lines]
    ys = [x['y1'] for x in lines] + [x['y2'] for x in lines]
    flags = []
    if min(xs) < 8 or max(xs) > (w - 8): flags.append('EDGE_TEXT_HORIZONTAL')
    if min(ys) < 8 or max(ys) > (h - 8): flags.append('EDGE_TEXT_VERTICAL')
    return flags

In [131]:
# CELL 13 — Extract a page
def extract_page(page_path, page_no):
    img_bgr, ocr_input, angle, quality, ocr_mode_used, raw_results = run_ocr(page_path)
    lines = ocr_to_lines(raw_results)
    inferred_type = infer_doc_type(lines) if DOC_TYPE == 'AUTO' else DOC_TYPE
    sections = build_sections(lines)
    header_lines = sections['header'] + sections['other']

    extracted = {
        'page_no': page_no, 'doc_type': inferred_type, 'text': '\n'.join([normalize_ocr_noise(x['text']) for x in lines]),
        'ocr_engine': ocr_mode_used, 'confidence': round(float(np.mean([x['confidence'] for x in lines])) if lines else 0.0, 4),
        'quality': {**quality, 'deskew_angle': round(angle,2)},
        'script_flags': sorted(list(set([s for s in [detect_script_flag(x['text']) for x in lines] if s]))),
        'fraud_checks': {'duplicate_stamp_terms': detect_duplicate_stamps(lines), 'alteration_flags': detect_document_alteration(lines), 'partial_flags': detect_partial_document(lines, img_bgr.shape), 'image_fingerprint': page_fingerprint(page_path)}
    }

    # Rich header fields
    for ln in header_lines[:10]:
        tx = normalize_ocr_noise(ln['text'])
        if 'dr.' in tx.lower() or tx.lower().startswith('dr '):
            extracted['doctor_name'] = {'value': tx.split('|')[0].strip(), 'confidence': round(float(ln['confidence']),3)}
            break
    reg = None
    for ln in lines:
        reg = validate_reg_no(ln['text'])
        if reg:
            extracted['doctor_reg_no'] = reg
            break
    specs = extract_specialization(header_lines)
    if specs: extracted['specialization'] = specs
    addr = extract_address(header_lines)
    if addr: extracted['address'] = addr
    gst = extract_gstin(lines)
    if gst: extracted['gstin'] = gst
    nabl = extract_nabl(lines)
    if nabl: extracted['nabl_status'] = nabl

    for field, prefixes in [('patient_name',['Patient Name','Patient','Pt','Name']), ('date',['Date']), ('sample_id',['Sample ID','Sample No','Accession']), ('bill_no',['Bill No','Invoice']), ('drug_license_no',['Drug Lic','DL No']), ('doctor_name_ref',['Ref Doctor','Referring Doctor','Dr'])]:
        val = named_entity(lines, prefixes)
        if val and field not in extracted:
            extracted[field] = val

    m, ln = find_first(lines, [r'Age[:\s]+(\d{1,3})', r'Age/Gender[:\s]+(\d{1,3})'])
    if m: extracted['age'] = {'value': int(m.group(1)), 'confidence': round(float(ln['confidence']),3)}
    m, ln = find_first(lines, [r'\b(Male|Female|M|F)\b'])
    if m:
        g = m.group(1)
        extracted['gender'] = {'value': 'Male' if g.upper()=='M' else 'Female' if g.upper()=='F' else g, 'confidence': round(float(ln['confidence']),3)}

    # Diagnosis
    dx_hits = []
    for ln in sections['diagnosis']:
        dv = best_vocab_match(ln['text'], diag_df, threshold=72, label='diagnosis')
        if dv:
            dx_hits.append({'raw_line': normalize_ocr_noise(ln['text']), 'canonical': dv['canonical'], 'code': dv['code'], 'match_score': dv['score'], 'match_method': dv['method'], 'confidence': round(float(ln['confidence']),3)})
    if dx_hits:
        extracted['diagnosis'] = sorted(dx_hits, key=lambda z: (z['match_score'], z['confidence']), reverse=True)[0]
    elif sections['diagnosis']:
        extracted['diagnosis_raw'] = ' | '.join([expand_diagnosis_shorthand(x['text']) for x in sections['diagnosis']])

    # Medicines
    meds = []
    med_lines = sections['rx'] if sections['rx'] else lines
    for ln in med_lines:
        raw = normalize_ocr_noise(ln['text'])
        rx = best_rxnorm_match(raw)
        if rx:
            meds.append({'raw_line': raw, 'canonical': rx['canonical'], 'rxcui': rx['rxcui'], 'tty': rx['tty'], 'sab': rx['sab'], 'match_score': rx['score'], 'match_method': rx['method'], 'core_query': rx['core_query'], 'dose': parse_dose(raw) if 'parse_dose' in globals() else None, 'frequency': None, 'duration': None, 'confidence': round(float(ln['confidence']),3)})
    if meds:
        extracted['medicines'] = pd.DataFrame(meds).sort_values(['match_score','confidence'], ascending=False).drop_duplicates(subset=['canonical']).to_dict(orient='records')
    else:
        extracted['medicines'] = []

    # Tests
    tests = []
    for ln in sections['tests']:
        for piece in re.split(r',|/|;|\band\b', normalize_ocr_noise(ln['text']), flags=re.I):
            piece = piece.strip()
            if not piece: continue
            tv = best_vocab_match(piece, lab_df, threshold=72, label='lab')
            if tv:
                tests.append({'raw_line': piece, 'canonical': tv['canonical'], 'code': tv['code'], 'match_score': tv['score'], 'match_method': tv['method'], 'confidence': round(float(ln['confidence']),3)})
    extracted['tests'] = pd.DataFrame(tests).sort_values(['match_score','confidence'], ascending=False).drop_duplicates(subset=['canonical']).to_dict(orient='records') if tests else []

    # Table extraction by doc type
    if inferred_type == 'HOSPITAL_BILL':
        extracted['line_items'] = parse_hospital_bill_rows(lines)
        m, _ = find_first(lines, [r'Subtotal[:\s]+([0-9,]+(?:\.[0-9]{1,2})?)']); extracted['subtotal'] = float(m.group(1).replace(',','')) if m else None
        m, _ = find_first(lines, [r'GST[^0-9]*([0-9,]+(?:\.[0-9]{1,2})?)']); extracted['gst_amount'] = float(m.group(1).replace(',','')) if m else None
        m, _ = find_first(lines, [r'Total Amount[:\s]+([0-9,]+(?:\.[0-9]{1,2})?)']); extracted['total_amount'] = float(m.group(1).replace(',','')) if m else None

    if inferred_type == 'PHARMACY_BILL':
        extracted['line_items'] = parse_pharmacy_rows(lines)
        m, _ = find_first(lines, [r'Subtotal[:\s]+([0-9,]+(?:\.[0-9]{1,2})?)'])
        extracted['subtotal'] = float(m.group(1).replace(',', '')) if m else None
        m, _ = find_first(lines, [r'Discount[^0-9-]*-?([0-9,]+(?:\.[0-9]{1,2})?)'])
        extracted['discount'] = float(m.group(1).replace(',', '')) if m else None
        m, _ = find_first(lines, [r'Net Amount[:\s]+([0-9,]+(?:\.[0-9]{1,2})?)'])
        extracted['net_amount'] = float(m.group(1).replace(',', '')) if m else None
        extracted['pharmacy_name'] = extracted.get('address') or (normalize_ocr_noise(lines[0]['text']) if lines else None)
        if not extracted.get('medicines') and extracted.get('line_items'):
            extracted['medicines'] = extracted['line_items']

    if inferred_type == 'LAB_REPORT':
        extracted['lab_rows'] = parse_lab_rows(lines)
        remarks = [normalize_ocr_noise(x['text']) for x in lines if re.search(r'remarks|clinical correlation', x['text'], re.I)]
        if remarks: extracted['remarks'] = remarks
        for ln in lines[::-1]:
            if 'dr.' in ln['text'].lower():
                extracted['pathologist_name'] = {'value': normalize_ocr_noise(ln['text']), 'confidence': round(float(ln['confidence']),3)}
                break

    if extracted['script_flags']:
        extracted['regional_fields_unextracted'] = True

    return extracted, lines

In [132]:
# CELL 14 — Minimal parsers missing from prior cells
def parse_dose(text):
    t = normalize_ocr_noise(text)
    m = re.search(r'(\d+(?:\.\d+)?\s*(?:mg|mcg|ml|g|gm|IU|units?|tabs?|caps?|drops?|sachet))', t, re.I)
    return m.group(1) if m else None

In [133]:
# CELL 15A — Tag validation and required fields
SUPPORTED_TAGS = {'PRESCRIPTION','HOSPITAL_BILL','LAB_REPORT','PHARMACY_BILL'}
REQUIRED_FIELDS = {
    'PRESCRIPTION': ['doctor_name','patient_name','date','medicines'],
    'HOSPITAL_BILL': ['hospital','bill_no','date','patient_name','line_items','total_amount'],
    'LAB_REPORT': ['lab_name','patient_name','sample_id','sample_date','report_date','tests'],
    'PHARMACY_BILL': ['pharmacy_name','bill_no','date','patient_name','medicines','net_amount']
}


def unwrap_value(v):
    if isinstance(v, dict) and 'value' in v:
        return v['value']
    return v


def is_present(v):
    v = unwrap_value(v)
    if v is None:
        return False
    if isinstance(v, str):
        return bool(v.strip())
    if isinstance(v, (list, tuple, dict, set)):
        return len(v) > 0
    return True


def compute_required_field_check(doc, expected_tag):
    req = REQUIRED_FIELDS.get(expected_tag, [])
    present, missing = [], []
    for f in req:
        if is_present(doc.get(f)):
            present.append(f)
        else:
            missing.append(f)
    score = round(len(present) / max(1, len(req)), 4)
    return {
        'expected_tag': expected_tag,
        'required_fields': req,
        'present_fields': present,
        'missing_fields': missing,
        'all_required_fields_present': len(missing) == 0,
        'required_field_coverage': score
    }


def build_tag_match_result(expected_tag, inferred_tag):
    ok = str(expected_tag).upper().strip() == str(inferred_tag).upper().strip()
    return {
        'expected_tag': expected_tag,
        'inferred_doc_type': inferred_tag,
        'tag_matches': ok,
        'message': 'Document matches expected tag.' if ok else f'Document type mismatch: expected {expected_tag}, inferred {inferred_tag}.'
    }


In [134]:
# CELL 15B — Continue V2 extractor with tag-aware schemas

def parse_dosage(text):
    t = normalize_ocr_noise(text)
    for pat in [r'\d-\d-\d', r'\d-\d ', r' SOS ', r' PRN ', r' OD ', r' BD ', r' TDS ', r' QID ', r' HS ', r' AC ', r' PC ']:
        m = re.search(pat, t, re.I)
        if m:
            return m.group(0).upper()
    return None


def parse_duration_days(text):
    t = normalize_ocr_noise(text)
    for pat, mult in [
        (r' x\s*(\d+)\s*(day|days) ', 1),
        (r' for\s*(\d+)\s*(day|days) ', 1),
        (r' x\s*(\d+)\s*(week|weeks) ', 7),
        (r' for\s*(\d+)\s*(week|weeks) ', 7),
        (r' x\s*(\d+)\s*(month|months) ', 30),
        (r' for\s*(\d+)\s*(month|months) ', 30),
    ]:
        m = re.search(pat, t, re.I)
        if m:
            return int(m.group(1)) * mult
    return None


def parse_note(text):
    t = normalize_ocr_noise(text)
    found = []
    for pat in [r' after food ', r' before food ', r' after meals ', r' before meals ', r' at bedtime ', r' if fever persists ', r' if pain persists ', r' as needed ', r' when necessary ']:
        m = re.search(pat, t, re.I)
        if m:
            found.append(m.group(0).strip().lower())
    return ' | '.join(dict.fromkeys(found)) if found else None


def clean_medicine_name_v3(raw_line, canonical=None):
    base = normalize_ocr_noise(canonical or raw_line)
    base = re.sub(r'^\d+[.)]\s*', '', base)
    base = re.sub(r' (tab|tablet|cap|capsule|syp|syrup|inj|injection|cream|ointment|gel|drops?|lotion|spray) ', '', base, flags=re.I)
    base = re.sub(r' (SOS|PRN|OD|BD|TDS|QID|HS|AC|PC) ', '', base, flags=re.I)
    base = re.sub(r' \d-\d-\d ', '', base)
    base = re.sub(r' \d-\d ', '', base)
    base = re.sub(r' x\s*\d+\s*(day|days|week|weeks|month|months) ', '', base, flags=re.I)
    base = re.sub(r' for\s*\d+\s*(day|days|week|weeks|month|months) ', '', base, flags=re.I)
    base = re.sub(r' after food | before food | after meals | before meals | at bedtime | if fever persists | if pain persists | as needed | when necessary ', '', base, flags=re.I)
    base = re.sub(r'\s+', ' ', base).strip(' -,:;')
    return base


def enrich_prescription_page(extracted, sections, lines):
    meds = []
    med_lines = sections['rx'] if sections['rx'] else lines
    for idx, ln in enumerate(med_lines, 1):
        raw = normalize_ocr_noise(ln['text'])
        rx = best_rxnorm_match(raw)
        if not (rx or looks_like_medicine_line(raw)):
            continue
        meds.append({
            'name': clean_medicine_name_v3(raw, rx['canonical'] if rx else None),
            'dosage': parse_dosage(raw),
            'duration_days': parse_duration_days(raw),
            'note': parse_note(raw),
            'raw_line': raw,
            'confidence': round(float(ln['confidence']), 3),
            'line_index': idx,
            'match_source': 'rxnorm' if rx else 'ocr_rule',
            'canonical': rx['canonical'] if rx else clean_medicine_name_v3(raw, None),
            'rxcui': rx['rxcui'] if rx else None,
            'match_score': rx['score'] if rx else None
        })
    if meds:
        extracted['medicines'] = pd.DataFrame(meds).drop_duplicates(subset=['name', 'dosage', 'duration_days', 'note', 'raw_line']).to_dict(orient='records')
    return extracted


def extract_pharmacy_name(lines):
    return normalize_ocr_noise(lines[0]['text']) if lines else None


def extract_hospital_name(lines):
    return normalize_ocr_noise(lines[0]['text']) if lines else None


def extract_lab_name(lines):
    return normalize_ocr_noise(lines[0]['text']) if lines else None


In [135]:
# CELL 16 — Process all pages, validate tag, check required fields, and build final JSON
assert EXPECTED_TAG in SUPPORTED_TAGS, f'EXPECTED_TAG must be one of {sorted(SUPPORTED_TAGS)}'

page_outputs = []
all_lines = []
for i, page in enumerate(page_files, start=1):
    ex, lines = extract_page(page, i)
    sections = build_sections(lines)

    # Continue V2 behavior with richer per-tag shaping
    if ex['doc_type'] == 'PRESCRIPTION':
        ex = enrich_prescription_page(ex, sections, lines)
    elif ex['doc_type'] == 'PHARMACY_BILL':
        ex['pharmacy_name'] = ex.get('pharmacy_name') or ex.get('hospital') or ex.get('address') or extract_pharmacy_name(lines)
        if not ex.get('medicines') and ex.get('line_items'):
            ex['medicines'] = ex['line_items']
    elif ex['doc_type'] == 'HOSPITAL_BILL':
        ex['hospital'] = ex.get('hospital') or ex.get('address') or extract_hospital_name(lines)
    elif ex['doc_type'] == 'LAB_REPORT':
        ex['lab_name'] = ex.get('lab_name') or ex.get('nabl_status') or extract_lab_name(lines)
        if not ex.get('tests') and ex.get('lab_rows'):
            ex['tests'] = [{'name': r.get('test_name'), 'result': r.get('result'), 'unit': r.get('unit'), 'range': r.get('normal_range')} for r in ex.get('lab_rows', [])]

    page_outputs.append(ex)
    all_lines.extend([dict(x, page_no=i) for x in lines])
    print('Processed page', i, '| inferred =', ex['doc_type'], '| OCR =', ex['ocr_engine'])

final = {
    'input_path': INPUT_PATH,
    'expected_tag': EXPECTED_TAG,
    'doc_type': page_outputs[0]['doc_type'] if page_outputs else 'UNKNOWN',
    'page_count': len(page_outputs),
    'pages': page_outputs,
    'aggregate_text': ''.join([p.get('text','') for p in page_outputs]),
    'quality_flags': sorted(list(set(sum([p['quality']['flags'] for p in page_outputs], [])))),
    'script_flags': sorted(list(set(sum([p.get('script_flags',[]) for p in page_outputs], []))))
}

for field in ['doctor_name','doctor_reg_no','specialization','address','patient_name','age','gender','date','sample_id','bill_no','drug_license_no','gstin','nabl_status','diagnosis','pathologist_name','hospital','lab_name','pharmacy_name','subtotal','discount','net_amount','gst_amount','total_amount','payment_mode','sample_date','report_date','remarks']:
    for p in page_outputs:
        if p.get(field) not in [None, '', [], {}]:
            final[field] = p[field]
            break

def merge_list_field(name, key='canonical'):
    rows = []
    for p in page_outputs:
        rows.extend(p.get(name, []))
    if not rows:
        return []
    df = pd.DataFrame(rows)
    return df.drop_duplicates(subset=[key]).to_dict(orient='records') if key in df.columns else rows

if final['doc_type'] == 'PRESCRIPTION':
    final['medicines'] = merge_list_field('medicines', 'name')
elif final['doc_type'] == 'PHARMACY_BILL':
    final['medicines'] = merge_list_field('medicines', 'name') if any(p.get('medicines') for p in page_outputs) else merge_list_field('line_items', 'medicine')
elif final['doc_type'] == 'HOSPITAL_BILL':
    final['line_items'] = merge_list_field('line_items', 'description')
elif final['doc_type'] == 'LAB_REPORT':
    final['tests'] = merge_list_field('tests', 'name') if any(p.get('tests') for p in page_outputs) else merge_list_field('lab_rows', 'test_name')

final['fraud_summary'] = {
    'duplicate_stamp_terms': sorted(list(set(sum([p['fraud_checks'].get('duplicate_stamp_terms',[]) for p in page_outputs], [])))),
    'alteration_flags': sorted(list(set(sum([p['fraud_checks'].get('alteration_flags',[]) for p in page_outputs], [])))),
    'partial_flags': sorted(list(set(sum([p['fraud_checks'].get('partial_flags',[]) for p in page_outputs], [])))),
    'duplicate_pages_detected': len([p['fraud_checks']['image_fingerprint'] for p in page_outputs]) != len(set([p['fraud_checks']['image_fingerprint'] for p in page_outputs]))
}
if final['fraud_summary']['alteration_flags']:
    final['DOCUMENT_ALTERATION'] = True
if final['script_flags']:
    final['regional_fields_unextracted'] = True

final['tag_validation'] = build_tag_match_result(EXPECTED_TAG, final['doc_type'])
final['required_field_validation'] = compute_required_field_check(final, EXPECTED_TAG)
final['ready_for_downstream_processing'] = final['tag_validation']['tag_matches'] and final['required_field_validation']['all_required_fields_present']

print(json.dumps({
    'tag_validation': final['tag_validation'],
    'required_field_validation': final['required_field_validation'],
    'ready_for_downstream_processing': final['ready_for_downstream_processing'],
    'doc_type': final['doc_type']
}, indent=2, ensure_ascii=False))


mode:  EASYOCR
Using EasyOCR directly in else block


d:\Interviews\Plum\Backend\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Processed page 1 | inferred = HOSPITAL_BILL | OCR = EASYOCR
{
  "tag_validation": {
    "expected_tag": "PHARMACY_BILL",
    "inferred_doc_type": "HOSPITAL_BILL",
    "tag_matches": false,
    "message": "Document type mismatch: expected PHARMACY_BILL, inferred HOSPITAL_BILL."
  },
  "required_field_validation": {
    "expected_tag": "PHARMACY_BILL",
    "required_fields": [
      "pharmacy_name",
      "bill_no",
      "date",
      "patient_name",
      "medicines",
      "net_amount"
    ],
    "present_fields": [
      "bill_no",
      "date",
      "patient_name"
    ],
    "missing_fields": [
      "pharmacy_name",
      "medicines",
      "net_amount"
    ],
    "all_required_fields_present": false,
    "required_field_coverage": 0.5
  },
  "ready_for_downstream_processing": false,
  "doc_type": "HOSPITAL_BILL"
}


In [136]:

# CELL 17 — Save outputs
with open(OUT_DIR / 'final_output.json', 'w', encoding='utf-8') as f:
    json.dump(final, f, indent=2, ensure_ascii=False)
with open(OUT_DIR / 'pages_output.json', 'w', encoding='utf-8') as f:
    json.dump(page_outputs, f, indent=2, ensure_ascii=False)
pd.DataFrame(all_lines).to_csv(OUT_DIR / 'all_lines.csv', index=False)
print('Saved outputs to', OUT_DIR)
print('Main file:', OUT_DIR / 'final_output.json')


Saved outputs to complete_pipeline_v3_output
Main file: complete_pipeline_v3_output\final_output.json


In [137]:
# # Final compact cell (V3): sample 20 images, run OCR, compare to ground truth, save combined JSON

# from pathlib import Path
# import json, random, re, pandas as pd

# GT_ROOT = Path("./phase_a_v3_full_suite/ground_truth")
# OUT_CSV = "./random20_eval_v3.csv"
# OUT_JSON = "./random20_eval_combined_v3.json"

# def nt(x): 
#     return re.sub(r"\s+", " ", str("" if x is None else x)).strip().lower()

# def nv(x):
#     return x.get("value", x.get("canonical", x)) if isinstance(x, dict) else x

# def nlist(x):
#     if not x:
#         return []
#     out = []
#     for i in x:
#         if isinstance(i, dict):
#             for k in ["name", "canonical", "description", "medicine", "test_name", "raw_line", "value"]:
#                 if i.get(k) not in [None, ""]:
#                     out.append(nt(i[k]))
#                     break
#         else:
#             out.append(nt(i))
#     return sorted(set(out))

# def lev(a, b):
#     a, b = nt(a), nt(b)
#     dp = list(range(len(b) + 1))
#     for i, ca in enumerate(a, 1):
#         prev, dp[0] = dp[0], i
#         for j, cb in enumerate(b, 1):
#             cur = dp[j]
#             dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (ca != cb))
#             prev = cur
#     return dp[-1]

# def cer(a, b):
#     a = nt(a)
#     return lev(a, b) / max(1, len(a))

# def wer(a, b):
#     a, b = nt(a).split(), nt(b).split()
#     dp = list(range(len(b) + 1))
#     for i, ca in enumerate(a, 1):
#         prev, dp[0] = dp[0], i
#         for j, cb in enumerate(b, 1):
#             cur = dp[j]
#             dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (ca != cb))
#             prev = cur
#     return dp[-1] / max(1, len(a))

# def cat(p):
#     p = str(p).upper()
#     if "HOSPITAL_BILL" in p: return "HOSPITAL_BILL"
#     if "LAB_REPORT" in p: return "LAB_REPORT"
#     if "PHARMACY_BILL" in p: return "PHARMACY_BILL"
#     return "PRESCRIPTION"

# def pp(s):
#     return str(s).replace("\\", "/").strip()

# gts = []
# for f in GT_ROOT.rglob("*.json"):
#     try:
#         gt = json.loads(f.read_text(encoding="utf-8"))
#         clean = pp(gt.get("clean_image_path", ""))
#         vars_ = [pp(v["variant_path"]) for v in gt.get("variants", []) if v.get("variant_path")]
#         chosen = random.choice([clean] + vars_) if (clean or vars_) else ""
#         gts.append({
#             "gt": gt,
#             "img": chosen,
#             "category": gt.get("doc_type", cat(chosen or clean))
#         })
#     except:
#         pass

# df = pd.DataFrame(gts)
# sample = pd.concat([
#     df[df.category == c].sample(min(5, len(df[df.category == c])), random_state=42)
#     for c in ["PRESCRIPTION", "HOSPITAL_BILL", "LAB_REPORT", "PHARMACY_BILL"]
# ], ignore_index=True).head(20)

# def run_one(img, expected_tag="AUTO"):
#     pages = render_pdf_pages(img) if str(img).lower().endswith(".pdf") else [img]
#     page_outputs = []

#     doc_type_override = globals().get("DOC_TYPE", "AUTO")

#     for i, p in enumerate(pages, 1):
#         ex, lines = extract_page(p, i)
#         sections = build_sections(lines)

#         if ex["doc_type"] == "PRESCRIPTION" and "enrich_prescription_page" in globals():
#             ex = enrich_prescription_page(ex, sections, lines)

#         elif ex["doc_type"] == "PHARMACY_BILL":
#             ex["pharmacy_name"] = ex.get("pharmacy_name") or ex.get("hospital") or ex.get("address") or (normalize_ocr_noise(lines[0]["text"]) if lines else None)
#             if not ex.get("medicines") and ex.get("line_items"):
#                 ex["medicines"] = ex["line_items"]

#         elif ex["doc_type"] == "HOSPITAL_BILL":
#             ex["hospital"] = ex.get("hospital") or ex.get("address") or (normalize_ocr_noise(lines[0]["text"]) if lines else None)

#         elif ex["doc_type"] == "LAB_REPORT":
#             ex["lab_name"] = ex.get("lab_name") or ex.get("nabl_status") or (normalize_ocr_noise(lines[0]["text"]) if lines else None)
#             if not ex.get("tests") and ex.get("lab_rows"):
#                 ex["tests"] = [
#                     {
#                         "name": r.get("test_name"),
#                         "result": r.get("result"),
#                         "unit": r.get("unit"),
#                         "range": r.get("normal_range")
#                     }
#                     for r in ex.get("lab_rows", [])
#                 ]

#         page_outputs.append(ex)

#     final = {
#         "expected_tag": expected_tag,
#         "doc_type": page_outputs[0]["doc_type"] if page_outputs else "UNKNOWN",
#         "aggregate_text": "\n\n".join([p.get("text", "") for p in page_outputs]),
#         "pages": page_outputs
#     }

#     for f in [
#         "doctor_name","doctor_reg_no","specialization","address","patient_name","age","gender","date",
#         "sample_id","bill_no","drug_license_no","gstin","nabl_status","diagnosis","pathologist_name",
#         "subtotal","gst_amount","total_amount","discount","net_amount","hospital","lab_name","pharmacy_name",
#         "remarks"
#     ]:
#         for p in page_outputs:
#             if p.get(f) not in [None, "", []]:
#                 final[f] = p[f]
#                 break

#     def merge(name, key):
#         rows = []
#         for p in page_outputs:
#             rows.extend(p.get(name, []))
#         if not rows:
#             return []
#         d = pd.DataFrame(rows)
#         return d.drop_duplicates(subset=[key]).to_dict(orient="records") if key in d.columns else rows

#     if final["doc_type"] == "PRESCRIPTION":
#         final["medicines"] = merge("medicines", "name") if any("name" in m for p in page_outputs for m in p.get("medicines", [])) else merge("medicines", "canonical")
#         final["tests"] = merge("tests", "canonical")
#         final["line_items"] = []
#         final["lab_rows"] = []

#     elif final["doc_type"] == "HOSPITAL_BILL":
#         final["medicines"] = []
#         final["tests"] = []
#         final["line_items"] = merge("line_items", "description")
#         final["lab_rows"] = []

#     elif final["doc_type"] == "PHARMACY_BILL":
#         final["medicines"] = merge("medicines", "name") if any("name" in m for p in page_outputs for m in p.get("medicines", [])) else merge("line_items", "medicine")
#         final["tests"] = []
#         final["line_items"] = merge("line_items", "medicine")
#         final["lab_rows"] = []

#     elif final["doc_type"] == "LAB_REPORT":
#         final["medicines"] = []
#         final["tests"] = merge("tests", "name") if any("name" in t for p in page_outputs for t in p.get("tests", [])) else merge("lab_rows", "test_name")
#         final["line_items"] = []
#         final["lab_rows"] = merge("lab_rows", "test_name")

#     else:
#         final["medicines"] = merge("medicines", "name") if any("name" in m for p in page_outputs for m in p.get("medicines", [])) else merge("medicines", "canonical")
#         final["tests"] = merge("tests", "canonical")
#         final["line_items"] = []
#         final["lab_rows"] = []

#     if "build_tag_match_result" in globals():
#         final["tag_validation"] = build_tag_match_result(expected_tag, final["doc_type"]) if expected_tag != "AUTO" else {
#             "expected_tag": expected_tag,
#             "inferred_doc_type": final["doc_type"],
#             "tag_matches": None,
#             "message": "AUTO mode used for evaluation."
#         }
#     else:
#         final["tag_validation"] = {
#             "expected_tag": expected_tag,
#             "inferred_doc_type": final["doc_type"],
#             "tag_matches": int(nt(expected_tag) == nt(final["doc_type"])) if expected_tag != "AUTO" else None
#         }

#     if "compute_required_field_check" in globals():
#         final["required_field_validation"] = compute_required_field_check(final, expected_tag) if expected_tag != "AUTO" else {}
#     else:
#         final["required_field_validation"] = {}

#     return final

# rows = []
# scalar_fields = [
#     "doctor_name","doctor_reg_no","specialization","address","patient_name","age","gender","date",
#     "sample_id","bill_no","drug_license_no","gstin","nabl_status","diagnosis","pathologist_name",
#     "subtotal","gst_amount","total_amount","discount","net_amount","hospital","lab_name","pharmacy_name"
# ]
# list_fields = ["medicines","tests","line_items","lab_rows"]

# for _, r in sample.iterrows():
#     gt, img = r["gt"], r["img"]
#     expected_tag = r["category"]

#     if not img or not Path(img).exists():
#         rows.append({
#             "image_path": img,
#             "category": expected_tag,
#             "doc_type_ok": 0,
#             "tag_match_ok": 0,
#             "required_fields_ok": 0,
#             "field_score": 0,
#             "text_cer": 1,
#             "text_wer": 1,
#             "error": "missing_image"
#         })
#         continue

#     pred = run_one(img, expected_tag=expected_tag)
#     exact = tot = 0

#     for f in scalar_fields:
#         gv, pv = gt.get(f), pred.get(f)
#         if gv is None and pv is None:
#             continue
#         tot += 1
#         exact += int(nt(nv(gv)) == nt(nv(pv)))

#     for f in list_fields:
#         gv, pv = gt.get(f), pred.get(f)
#         if not gv and not pv:
#             continue
#         tot += 1
#         exact += int(set(nlist(gv)) == set(nlist(pv)))

#     rows.append({
#         "image_path": img,
#         "category": expected_tag,
#         "doc_type_ok": int(nt(gt.get("doc_type", expected_tag)) == nt(pred.get("doc_type"))),
#         "tag_match_ok": int(pred.get("tag_validation", {}).get("tag_matches") is True) if pred.get("tag_validation", {}).get("tag_matches") is not None else 0,
#         "required_fields_ok": int(pred.get("required_field_validation", {}).get("all_required_fields_present", False)),
#         "field_score": round(exact / max(1, tot), 4),
#         "text_cer": round(cer(gt.get("aggregate_text", gt.get("text", "")), pred.get("aggregate_text", "")), 4),
#         "text_wer": round(wer(gt.get("aggregate_text", gt.get("text", "")), pred.get("aggregate_text", "")), 4),
#         "ground_truth": gt,
#         "prediction": pred
#     })

# rdf = pd.DataFrame(rows)

# results = {
#     "total_docs": len(rows),
#     "docs_evaluated": [
#         {
#             "image_path": r["image_path"],
#             "category": r["category"],
#             "prediction": r["prediction"],
#             "ground_truth": r["ground_truth"],
#             "metrics": {
#                 "doc_type_ok": r["doc_type_ok"],
#                 "tag_match_ok": r["tag_match_ok"],
#                 "required_fields_ok": r["required_fields_ok"],
#                 "field_score": r["field_score"],
#                 "text_cer": r["text_cer"],
#                 "text_wer": r["text_wer"]
#             }
#         }
#         for r in rows
#     ],
#     "summary": {
#         "total": len(rows),
#         "doc_type_accuracy": round(rdf["doc_type_ok"].mean(), 4) if len(rdf) else 0.0,
#         "tag_match_accuracy": round(rdf["tag_match_ok"].mean(), 4) if len(rdf) else 0.0,
#         "required_fields_pass_rate": round(rdf["required_fields_ok"].mean(), 4) if len(rdf) else 0.0,
#         "mean_field_score": round(rdf["field_score"].mean(), 4) if len(rdf) else 0.0,
#         "mean_text_cer": round(rdf["text_cer"].mean(), 4) if len(rdf) else 0.0,
#         "mean_text_wer": round(rdf["text_wer"].mean(), 4) if len(rdf) else 0.0
#     }
# }

# with open(OUT_JSON, "w", encoding="utf-8") as f:
#     json.dump(results, f, indent=2, ensure_ascii=False)

# pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
# print("SAVED:", OUT_JSON, OUT_CSV)
# print("\nSUMMARY")
# print(json.dumps(results["summary"], indent=2))